In [ ]:
pip install torch

In [ ]:

import torch
from torch import linalg as LA
import os
import time
import statistics
from typing import Tuple
from typing_extensions import Annotated

In [ ]:
import os, platform, re, subprocess, sys
import psutil

def cpu_name():
    if sys.platform == "darwin":
        return subprocess.run(["sysctl", "-n", "machdep.cpu.brand_string"],
                              capture_output=True, text=True).stdout.strip()
    if sys.platform.startswith("linux"):
        with open("/proc/cpuinfo") as f:
            for line in f:
                if line.startswith("model name"):
                    return line.split(":", 1)[1].strip()
    return platform.processor()

print("chip:", cpu_name())
print("physical cores:", psutil.cpu_count(logical=False))
print("logical cores:", psutil.cpu_count(logical=True))
print("memory (GB):", psutil.virtual_memory().total / 2**30)
print("OS:", platform.platform(), platform.machine())
print("python:", sys.executable)
print("torch:", torch.__version__, re.findall(r"BLAS_INFO=\w+", torch.__config__.show()))

In [ ]:
torch.manual_seed(42)
a= torch.randn(500000000,device="cpu")


In [ ]:
def exp(a:torch.tensor)->torch.tensor:
    expo=torch.exp(a)
    return expo


In [ ]:
def time_counter(no_of_threads:int,a:torch.tensor,results:list):
    n_dict={}
    n_lst=[]

    torch.set_num_threads(no_of_threads)
    print(torch.get_num_threads())
    time.sleep(2)
    exp(a)
    for i in range(10):
        
        start_time=time.perf_counter()
        exp(a)
        end_time=time.perf_counter()
        n_lst.append(end_time-start_time)
    mean=statistics.mean(n_lst)
    stndev=statistics.stdev(n_lst)
    n_dict['thread']=torch.get_num_threads()
    n_dict['mean']=mean
    n_dict['stdev']=stndev
  
    results.append(n_dict)
    

    

In [ ]:
results=[]
for i in range(1,13):
    time_counter(i,a,results)


In [ ]:
results

In [ ]:
print(torch.__config__.show())

In [ ]:
n_results=[{'thread': 1, 'mean': 0.3810020999000002, 'stdev': 0.007944879141332005},
 {'thread': 2, 'mean': 0.19075557529999826, 'stdev': 0.0005270187960481285},
 {'thread': 3, 'mean': 0.13120751259999963, 'stdev': 0.00246158431518823},
 {'thread': 4, 'mean': 0.0993194459999998, 'stdev': 0.0011303969261131433},
 {'thread': 5, 'mean': 0.07783838330000066, 'stdev': 0.00039813339827680596},
 {'thread': 6, 'mean': 0.0797178123000009, 'stdev': 0.00352745778296632},
 {'thread': 7, 'mean': 0.0720301080999974, 'stdev': 0.007706562256238033},
 {'thread': 8, 'mean': 0.06401090009999706, 'stdev': 0.004476362159011306},
 {'thread': 9, 'mean': 0.05884946659999883, 'stdev': 0.002214189923585331},
 {'thread': 10, 'mean': 0.05733210819999925, 'stdev': 0.002203077830259051},
 {'thread': 11, 'mean': 0.057241833300001585, 'stdev': 0.0023205429242513847},
 {'thread': 12, 'mean': 0.06054750010000305, 'stdev': 0.00437113729273258}]

t1=n_results[0]['mean']
n_res=[]
for i in range(len(n_results)):
    n_s_e={}
    speedup=t1/n_results[i]['mean']
    efficiency=(speedup/n_results[i]['thread'])*100
    n_s_e['threads']=n_results[i]['thread']
    n_s_e['speedup']=speedup
    n_s_e['efficiency']=efficiency
    n_res.append(n_s_e)





In [ ]:
n_res

In [ ]:
pip install matplotlib

In [ ]:
import matplotlib.pyplot as plt

RUN_NAME = "run4"   # change per run, used in the file names

threads = [r["threads"] for r in n_res]
speedup = [r["speedup"] for r in n_res]
efficiency = [r["efficiency"] for r in n_res]
# Amdahl serial fraction (%), skipping 1 thread
f_threads = [n for n in threads if n > 1]
serial = [((n / s) - 1) / (n - 1) * 100 for n, s in zip(threads, speedup) if n > 1]

BLUE, GREY = "#2a78d6", "#8a8980"

def tidy(ax, ylabel, title):
    ax.set_xlabel("Threads (N)")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.set_xticks(threads)
    ax.grid(True, color="#e4e3dc")
    ax.spines[["top", "right"]].set_visible(False)

# 1. Speedup vs ideal
fig, ax = plt.subplots(figsize=(6.4, 4.2))
ax.plot(threads, threads, "--", color=GREY, label="Ideal (S = N)")
ax.plot(threads, speedup, "o-", color=BLUE, linewidth=2, label="Measured")
tidy(ax, "Speedup (T1 / TN)", "Speedup vs threads")
ax.legend(frameon=False)
fig.tight_layout(); fig.savefig(f"{RUN_NAME}_speedup.png", dpi=200); plt.show()

# 2. Efficiency
fig, ax = plt.subplots(figsize=(6.4, 4.2))
ax.plot(threads, efficiency, "o-", color=BLUE, linewidth=2)
ax.set_ylim(0, 110)
tidy(ax, "Parallel efficiency (%)", "Parallel efficiency vs threads")
fig.tight_layout(); fig.savefig(f"{RUN_NAME}_efficiency.png", dpi=200); plt.show()

# 3. Serial fraction
fig, ax = plt.subplots(figsize=(6.4, 4.2))
ax.plot(f_threads, serial, "o-", color=BLUE, linewidth=2)
ax.set_ylim(bottom=0)
tidy(ax, "Implied serial fraction f (%)", "Amdahl serial fraction vs threads")
fig.tight_layout(); fig.savefig(f"{RUN_NAME}_serial_fraction.png", dpi=200); plt.show()

----------------------------------------------------------------------------------------------------------------
What did you test?
A-  avg timed operation (of 10 repeats ) of exp (e to the power of x) applied to each number in a tensor, with x=500,000,000, on CPU threads in range (1,12), 12 being the maximum cores in M3pro chip (6 performance + 6 efficency) cores
-----------------------------------------------------------------------------------------------------------------

What happened?

Speedup & Efficiency Analysis
[{'threads': 1, 'speedup': 1.0, 'efficiency': 100.0},
 {'threads': 2,
  'speedup': 1.9088002575387968,
  'efficiency': 95.44001287693985},
 {'threads': 3, 'speedup': 2.720785212026601, 'efficiency': 90.6928404008867},
 {'threads': 4,
  'speedup': 3.6414543821125536,
  'efficiency': 91.03635955281383},
 {'threads': 5, 'speedup': 4.516644876500827, 'efficiency': 90.33289753001654},
 {'threads': 6, 'speedup': 4.84207431197496, 'efficiency': 80.701238532916},
 {'threads': 7, 'speedup': 5.023458866995003, 'efficiency': 71.7636980999286},
 {'threads': 8, 'speedup': 5.367714351977242, 'efficiency': 67.09642939971553},
 {'threads': 9, 'speedup': 5.432789966089792, 'efficiency': 60.36433295655325},
 {'threads': 10,
  'speedup': 5.652173741014499,
  'efficiency': 56.52173741014499},
 {'threads': 11,
  'speedup': 5.560355212088323,
  'efficiency': 50.54868374625748},
 {'threads': 12,
  'speedup': 5.469736621176738,
  'efficiency': 45.581138509806145}]
  as we can see the efficiency always decreases when we add newer threads. effiency means it is usinthe full capacity and capability of threads. so we can establish that efficiency decreases when we add threads, with one small rise from 3 to 4 threads (0.3 percentage points). 
  from the results we can establishthe fact that from threads 1-5 the efficency is fine that is above 90%, but in the 6the thread there is a massive decrease in efficiency at a  rate of 10 percentage points per thread and then it it again decreases slowly again at a rate of 4-5 percentage points.
  As for the speedup yes from threads 1-5 there is massive jump of speedup at an avg rate of 1+ times from 5-6 and 6-10 the speedup stalls but increases slowly at avg rate of  under the half from the first 5. but from threads 11-10 it actually little bit decreases and decreases more further.

-----------------------------------------------------------------------------------------------------------------

Standard deviation is a statistical measure that quantifies the amount of dispersion, spread, or variation in a set of data values relative to its mean (average).

Key Concepts

• Low Standard Deviation: Indicates that the data points tend to be very close or clustered tightly to the mean.
• High Standard Deviation: Indicates that the data points are spread out over a much wider range of values.
• Same Units: Unlike variance (which squares the differences), the standard deviation is expressed in the exact same unit of measurement as the original data, making it easier to interpret.
-----------------------------------------------------------------------------------------------------------------

How reliable is it?

from threads 1-5 the standard deviation becomes low even though there is an anomaly in thread 2. Low standatandrd of devaiation means most of the 10 executions are similar then it gradually increases from thread 6-12 so the exec times are slighlty varird for thr 10 repeats for each threads, we can say the data from 1-5 threads ae pretty reilable compared to the next threads


-----------------------------------------------------------------------------------------------------------------

Why did it happen?

Amdhals law - It  means that the speedup is limited by the serial (non-parallelisable) fraction of the workload  of CPUs than the parallizible parts

 f- The serial fraction
 N-number of threads
 S-Speedup
  S=1/(f+[(1-f)/N])
  f=(N/S)-1/(N-1)
 Threads (N)	Speedup (S)	Serial fraction (f)
    1	         1.00	       (baseline)
    2	         1.91	       4.78%
    3	         2.72	       5.13%
    4	         3.64	       3.28%
    5	         4.52	       2.68%
    6	         4.84	       4.80%
    7	         5.03	       6.55%
    8	         5.37	       6.99%
    9	         5.43	       8.22%
    10	       5.66	       8.53%
    11	       5.56	       9.79%
    12	       5.47	       10.85%


  According to the theory the serial part should be constant but after assesing the numbers we can see that the from thread 2-3 it remains similar  but it drops for the next few threads and slowly rises until the end. This might be due to 6 performace and 6 efficency cores. The macos decides what work to assign to which core so the most probable hypothesis is until thread 5 perfornce cores were added and then on the rise in f is due to the addition of efficiency cores.A likely reason is that background processes were using part of a performance core in run 1. In run 3, with fewer background processes, efficiency stayed near 89% up to 6 threads and only dropped at 7.
  Roofline
   is the mean time  0.110821
   Step 1: bytes per call
   bytes per call= 500,000,000 × 4 bytes = 2 GB
   write out put= 500,000,000 × 4 bytes = 2 GB
   approx 4B per call
   Step 2: achieved bandwidth
   4 GB ÷ your run 1 mean time at 12 threads (0.0643 s) =62.2 GB/s.
   Step 3: arithmetic intensity
   FLOPS=500,000,000 × 15=7500000000
   AI = FLOPs ÷ 4,000,000,000 bytes =1.875 flops/byte
   Step 4: compare
   62.2/150%(peak m3pro bandwidth)




  ---------------------------------------------------------------------------------------------------------------


In [ ]:
pip install datasets